# Figuras del capítulo 5 — Detección y evaluación (OE2)

Generado al dividir `notebooks/figuras_documento.ipynb` por capítulo.
Cada celda es independiente después de la sección 0: se puede correr solo la que cambió.


## 0. Preparación: rutas, estilo y helpers

Las figuras se escriben en `../figures/` de este mismo capítulo.


In [ ]:
import json
import logging
import sys
import textwrap
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch, PathPatch, Rectangle
from matplotlib.path import Path as MplPath

NB_DIR = Path.cwd()
CHAPTER_DIR = NB_DIR.parent
PROJECT_DIR = next(d for d in (NB_DIR, *NB_DIR.parents) if (d / "pyproject.toml").exists())
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from core.config import CLEANED_DIR, PROCESSED_DIR, RAW_DIR, RUNS_DIR, P  # noqa: E402

logging.getLogger("data.annotations").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

FIGURE_DIR = CHAPTER_DIR / "figures"  # capitulos/<capitulo>/figures
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = PROCESSED_DIR
COMPARISON_DIR = RUNS_DIR / "comparacion"

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [ ]:
# --- Estilo ------------------------------------------------------------------
SURFACE = "#ffffff"
INK = "#0b0b0b"
INK_2 = "#525252"
MUTED = "#8a8a8a"
GRID = "#e3e3e3"
BASELINE = "#c4c4c4"

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # slots categóricos 1-4
CRITICAL = "#d03b3b"
WARNING = "#eda100"
GOOD = "#1baf7a"
BLUE = "#2a78d6"
BLUE_FADED = "#b7d3f6"
BLUE_INK = "#0d366b"
PANEL = "#f3f3f3"  # relleno de caja de diagrama

mpl.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "savefig.edgecolor": SURFACE,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 9,
        "text.color": INK,
        "axes.edgecolor": BASELINE,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 10,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "legend.frameon": False,
        "legend.fontsize": 8.5,
        "lines.linewidth": 2,
    }
)


def style_value_axis(ax, axis="x"):
    """Rejilla fina sólo en el eje de valor; sin espina de línea base en ese eje."""
    ax.grid(True, axis=axis, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    if axis == "x":
        ax.spines["bottom"].set_visible(False)
        ax.spines["left"].set_color(BASELINE)
    else:
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_color(BASELINE)


_K = 0.5523  # constante de control del arco circular en una Bezier cúbica


def _square_path(x0, y0, w, h):
    return MplPath(
        [(x0, y0), (x0 + w, y0), (x0 + w, y0 + h), (x0, y0 + h), (x0, y0)],
        [MplPath.MOVETO] + [MplPath.LINETO] * 3 + [MplPath.CLOSEPOLY],
    )


def _rounded_path(x0, y0, w, h, rx, ry, side="right"):
    """Rectángulo con las dos esquinas de `side` redondeadas: el extremo de dato."""
    rx, ry = min(abs(rx), abs(w)), min(abs(ry), abs(h) / 2)
    if rx <= 0 or ry <= 0 or w == 0 or h == 0:
        return _square_path(x0, y0, w, h)
    x1, y1 = x0 + w, y0 + h

    if side == "right":
        verts = [
            (x0, y0),
            (x1 - rx, y0),
            (x1 - rx + rx * _K, y0),
            (x1, y0 + ry - ry * _K),
            (x1, y0 + ry),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0, y1),
            (x0, y0),
        ]
    else:  # "top"
        verts = [
            (x0, y0),
            (x1, y0),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0 + rx, y1),
            (x0 + rx - rx * _K, y1),
            (x0, y1 - ry + ry * _K),
            (x0, y1 - ry),
            (x0, y0),
        ]
    codes = [
        MplPath.MOVETO,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CLOSEPOLY,
    ]
    return MplPath(verts, codes)


def _x_per_y(ax):
    """Unidades de x que se dibujan tan largas como una unidad de y."""
    box = ax.get_position()
    width_in = ax.figure.get_figwidth() * box.width
    height_in = ax.figure.get_figheight() * box.height
    x_range = abs(np.diff(ax.get_xlim())[0])
    y_range = abs(np.diff(ax.get_ylim())[0])
    return (x_range / width_in) / (y_range / height_in)


def barh(ax, ys, widths, color, height=0.62, left=None, radius_frac=0.30, **kw):
    """Barras horizontales con el extremo de dato redondeado. `left` apila."""
    ys = np.atleast_1d(np.asarray(ys, dtype=float))
    widths = np.asarray(widths, dtype=float)
    left = np.zeros_like(widths) if left is None else np.asarray(left, dtype=float)
    ry = height * radius_frac
    rx = ry * _x_per_y(ax)
    for y, w, x0 in zip(ys, widths, left):
        ax.add_patch(
            PathPatch(
                _rounded_path(x0, y - height / 2, w, height, rx, ry, "right"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


def barv(ax, xs, heights, color, width=0.68, radius_frac=0.30, bottom=None, **kw):
    """Barras verticales con la punta redondeada."""
    xs = np.atleast_1d(np.asarray(xs, dtype=float))
    heights = np.asarray(heights, dtype=float)
    bottom = np.zeros_like(heights) if bottom is None else np.asarray(bottom, dtype=float)
    rx = width * radius_frac
    ry = rx / _x_per_y(ax)
    for x, h, y0 in zip(xs, heights, bottom):
        ax.add_patch(
            PathPatch(
                _rounded_path(x - width / 2, y0, width, h, rx, ry, "top"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


# --- Salida ------------------------------------------------------------------
# Dos artefactos por figura: el PNG que `main.tex` incluye hoy y, cuando la figura es
# una serie de datos y no un diagrama, el JSON con esos datos, para poder redibujarla
# en pgfplots sin volver a correr el cuaderno.
DATA_DIR = FIGURE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _jsonable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (pd.Series, pd.Index)):
        return value.tolist()
    if isinstance(value, pd.DataFrame):
        return value.to_dict(orient="list")
    raise TypeError(type(value))


def save(fig, name, data=None):
    path = FIGURE_DIR / f"{name}.png"
    fig.savefig(path)
    if data is not None:
        target = DATA_DIR / f"{name}.json"
        target.write_text(json.dumps(data, indent=1, ensure_ascii=False, default=_jsonable))

In [ ]:
# --- Helpers de diagrama ------------------------------------------------------


def canvas(width_in=10.0, height_in=5.5):
    fig, ax = plt.subplots(figsize=(width_in, height_in))
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100 * height_in / width_in)
    ax.set_aspect("equal")
    ax.axis("off")
    return fig, ax


def wrap_text(text, width=None):
    """Ajusta a `width` respetando los saltos de línea que ya trae el texto."""
    text = str(text)
    if not width:
        return text
    return "\n".join(
        textwrap.fill(line, width) if line.strip() else line for line in text.split("\n")
    )


def _units_per_point(ax):
    """Unidades del lienzo que mide un punto tipográfico (lienzo de 100 de ancho)."""
    return 100.0 / (ax.figure.get_figwidth() * 72.0)


def _stack(ax, blocks):
    """Alturas y separaciones de una pila de bloques de texto, en unidades del lienzo."""
    upt = _units_per_point(ax)
    heights = [(text.count("\n") + 1) * size * 1.34 * upt for text, size, _, _ in blocks]
    gaps = [1.15 * (blocks[i][1] + blocks[i + 1][1]) / 2 * upt for i in range(len(blocks) - 1)]
    return heights, gaps, sum(heights) + sum(gaps)


def _blocks(
    title, body, foot, fs, body_fs, foot_fs, ink, body_ink, foot_ink, weight, wrap, body_wrap
):
    blocks = []
    if title:
        blocks.append((wrap_text(title, wrap), fs, ink, weight))
    if body:
        blocks.append((wrap_text(body, body_wrap or wrap or 30), body_fs, body_ink, "normal"))
    if foot:
        blocks.append((wrap_text(foot, body_wrap or wrap or 30), foot_fs, foot_ink, "normal"))
    return blocks


def node(
    ax,
    cx,
    cy,
    w,
    h=None,
    title="",
    body=None,
    foot=None,
    *,
    fill=PANEL,
    edge=BASELINE,
    ink=INK,
    body_ink=INK_2,
    foot_ink=MUTED,
    lw=1.1,
    fs=9.0,
    body_fs=7.8,
    foot_fs=7.0,
    weight="bold",
    dashed=False,
    wrap=None,
    body_wrap=None,
    radius=1.4,
    pad_pt=9.0,
    zorder=2,
):
    """Caja redondeada centrada en (cx, cy) con hasta tres bloques de texto apilados.

    Con `h=None` la caja se ajusta al texto: es lo que evita las cajas medio vacías y
    los títulos encimados cuando el contenido cambia de largo.
    """
    blocks = _blocks(
        title, body, foot, fs, body_fs, foot_fs, ink, body_ink, foot_ink, weight, wrap, body_wrap
    )
    heights, gaps, total = _stack(ax, blocks)
    if h is None:
        h = total + 2 * pad_pt * _units_per_point(ax)

    ax.add_patch(
        FancyBboxPatch(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            boxstyle=f"round,pad=0,rounding_size={radius}",
            linewidth=lw,
            edgecolor=edge,
            facecolor=fill,
            linestyle=(0, (4, 3)) if dashed else "-",
            zorder=zorder,
            mutation_aspect=1,
        )
    )

    y = cy + total / 2
    for i, (text, size, color, wt) in enumerate(blocks):
        ax.text(
            cx,
            y - heights[i] / 2,
            text,
            ha="center",
            va="center",
            fontsize=size,
            color=color,
            fontweight=wt,
            zorder=zorder + 1,
            linespacing=1.32,
        )
        y -= heights[i] + (gaps[i] if i < len(gaps) else 0)
    return cx, cy, w, h


def node_height(
    ax,
    title,
    body=None,
    foot=None,
    *,
    fs=9.0,
    body_fs=7.8,
    foot_fs=7.0,
    wrap=None,
    body_wrap=None,
    pad_pt=9.0,
):
    blocks = _blocks(
        title, body, foot, fs, body_fs, foot_fs, INK, INK_2, MUTED, "bold", wrap, body_wrap
    )
    return _stack(ax, blocks)[2] + 2 * pad_pt * _units_per_point(ax)


def flow_row(ax, items, cy, *, margin=2.5, gap=2.6, h=None, arrow_color=BLUE, arrows=True, **kw):
    """Fila de cajas que reparte el ancho del lienzo, con flechas entre ellas.

    `items` es una lista de dicts con `title` y, opcionalmente, `body`, `foot` y `kw`.
    El alto es común a toda la fila: el de la caja que más texto lleva.
    """
    n = len(items)
    w = (100 - 2 * margin - (n - 1) * gap) / n
    if h is None:
        h = max(
            node_height(
                ax,
                item["title"],
                item.get("body"),
                item.get("foot"),
                fs=kw.get("fs", 9.0),
                body_fs=kw.get("body_fs", 7.8),
                foot_fs=kw.get("foot_fs", 7.0),
                wrap=item.get("kw", {}).get("wrap", kw.get("wrap")),
                body_wrap=kw.get("body_wrap"),
            )
            for item in items
        )
    boxes = []
    for i, item in enumerate(items):
        cx = margin + w / 2 + i * (w + gap)
        options = dict(kw)
        options.update(item.get("kw", {}))
        boxes.append(
            node(ax, cx, cy, w, h, item["title"], item.get("body"), item.get("foot"), **options)
        )
        if arrows and i:
            arrow(ax, (cx - w / 2 - gap, cy), (cx - w / 2, cy), color=arrow_color, lw=1.3)
    return boxes


def arrow(ax, start, end, *, color=MUTED, lw=1.2, rad=0.0, dashed=False, head="-|>", zorder=1):
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        zorder=zorder,
        arrowprops=dict(
            arrowstyle=head,
            color=color,
            linewidth=lw,
            shrinkA=1.5,
            shrinkB=1.5,
            linestyle=(0, (4, 3)) if dashed else "-",
            connectionstyle=f"arc3,rad={rad}",
            mutation_scale=11,
        ),
    )


def elbow(ax, points, *, color=BLUE, lw=1.3):
    """Conector en ángulo recto con punta de flecha en el último tramo."""
    xs, ys = zip(*points)
    ax.plot(xs, ys, color=color, linewidth=lw, solid_capstyle="round", zorder=1)
    arrow(ax, points[-2], points[-1], color=color, lw=lw)


def caption_at(ax, x, y, text, *, fs=7.8, color=INK_2, ha="center", va="center", wrap=None, **kw):
    ax.text(
        x,
        y,
        wrap_text(text, wrap),
        ha=ha,
        va=va,
        fontsize=fs,
        color=color,
        linespacing=1.35,
        **kw,
    )


def chip(ax, cx, cy, text, color=BLUE, *, fs=7.5, pad=1.0, ink=SURFACE):
    """Etiqueta pequeña de fondo lleno: para marcar una salida o un estado."""
    ax.text(
        cx,
        cy,
        text,
        ha="center",
        va="center",
        fontsize=fs,
        color=ink,
        fontweight="bold",
        zorder=5,
        bbox=dict(boxstyle=f"round,pad={pad * 0.3}", facecolor=color, edgecolor="none"),
    )


# --- Glifos de tensor ---------------------------------------------------------
# `node` dibuja una caja con texto: sirve para un módulo, no para el dato que viaja
# entre módulos. Esto es el vocabulario que faltaba -- el espectrograma como plano, los
# tokens contables, la pirámide a escala -- para que un diagrama muestre la forma del
# tensor en vez de describirla por escrito.


def slab(ax, cx, cy, w, h, *, skew=0.16, fill=SURFACE, edge=INK, lw=1.2, zorder=2):
    """Plano en perspectiva: el espectrograma y los mapas de salida."""
    dx = w * skew
    points = [
        (cx - w / 2 + dx, cy - h / 2),
        (cx + w / 2, cy - h / 2),
        (cx + w / 2 - dx, cy + h / 2),
        (cx - w / 2, cy + h / 2),
    ]
    ax.add_patch(
        plt.Polygon(points, closed=True, facecolor=fill, edgecolor=edge, lw=lw, zorder=zorder)
    )
    return points


def token_strip(
    ax, cx, cy, w, h, n, *, vertical=True, fill=SURFACE, edge=INK, lw=0.9, gap=0.30, zorder=3
):
    """Tokens dibujados de a uno: se ven y se cuentan, no se leen en un pie de caja."""
    if vertical:
        cell = h / n
        for k in range(n):
            ax.add_patch(
                Rectangle(
                    (cx - w / 2, cy - h / 2 + k * cell + gap / 2),
                    w,
                    cell - gap,
                    facecolor=fill,
                    edgecolor=edge,
                    lw=lw,
                    zorder=zorder,
                )
            )
    else:
        cell = w / n
        for k in range(n):
            ax.add_patch(
                Rectangle(
                    (cx - w / 2 + k * cell + gap / 2, cy - h / 2),
                    cell - gap,
                    h,
                    facecolor=fill,
                    edgecolor=edge,
                    lw=lw,
                    zorder=zorder,
                )
            )


def feature_pyramid(
    ax, cx, cy, w, h, levels, *, fill=SURFACE, edge=INK, lw=1.1, label_y=None, zorder=3
):
    """Un plano por nivel, con el alto proporcional a su resolución.

    La proporción entre los niveles *es* el dato: dibujarlos todos iguales borra lo
    único que distingue una pirámide multiescala de tres convoluciones en fila.
    """
    tallest = max(rows for rows, _ in levels)
    slot = w / len(levels)
    for k, (rows, cols) in enumerate(levels):
        x = cx - w / 2 + slot * (k + 0.5)
        height = h * (rows / tallest) ** 0.55
        ax.add_patch(
            Rectangle(
                (x - slot * 0.22, cy - height / 2),
                slot * 0.44,
                height,
                facecolor=fill,
                edgecolor=edge,
                lw=lw,
                zorder=zorder,
            )
        )
        y = cy - h / 2 - 1.4 if label_y is None else label_y
        caption_at(ax, x, y, f"{rows}x{cols}", fs=6.0, color=MUTED, va="top")


def patch_grid(ax, cx, cy, w, h, n_cols, n_rows, *, edge=INK, lw=0.9, zorder=3):
    """La rejilla de parches sobre la entrada: el parcheo del AST hecho visible."""
    ax.add_patch(
        Rectangle(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            facecolor=SURFACE,
            edgecolor=edge,
            lw=lw,
            zorder=zorder,
        )
    )
    for i in range(1, n_cols):
        x = cx - w / 2 + w * i / n_cols
        ax.plot([x, x], [cy - h / 2, cy + h / 2], color=edge, lw=lw * 0.7, zorder=zorder + 1)
    for j in range(1, n_rows):
        y = cy - h / 2 + h * j / n_rows
        ax.plot([cx - w / 2, cx + w / 2], [y, y], color=edge, lw=lw * 0.7, zorder=zorder + 1)


def detection_slab(ax, cx, cy, w, h, boxes, *, color=CRITICAL, **kw):
    """El plano de salida con las cajas encima: dónde termina el pipeline."""
    slab(ax, cx, cy, w, h, **kw)
    for bx, by, bw, bh in boxes:
        ax.add_patch(
            Rectangle(
                (cx - w / 2 + (bx - bw / 2) * w, cy - h / 2 + (by - bh / 2) * h),
                bw * w,
                bh * h,
                facecolor="none",
                edgecolor=color,
                lw=1.3,
                zorder=6,
            )
        )


def zoom_arrow(ax, cx, cy, w, h, *, fill=PANEL, edge=MUTED, lw=1.0, zorder=1):
    """Flecha hueca hacia abajo: 'esto de arriba se abre en este panel'."""
    shaft, head = w * 0.42, h * 0.46
    points = [
        (cx - shaft / 2, cy + h / 2),
        (cx + shaft / 2, cy + h / 2),
        (cx + shaft / 2, cy - h / 2 + head),
        (cx + w / 2, cy - h / 2 + head),
        (cx, cy - h / 2),
        (cx - w / 2, cy - h / 2 + head),
        (cx - shaft / 2, cy - h / 2 + head),
    ]
    ax.add_patch(
        plt.Polygon(points, closed=True, facecolor=fill, edgecolor=edge, lw=lw, zorder=zorder)
    )


def detail_panel(ax, cx, cy, w, h, title="", *, edge=BASELINE, lw=1.1, fs=7.6, zorder=0):
    """El recuadro que enmarca un panel de detalle del segundo piso."""
    ax.add_patch(
        FancyBboxPatch(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            boxstyle="round,pad=0,rounding_size=1.6",
            facecolor=SURFACE,
            edgecolor=edge,
            lw=lw,
            zorder=zorder,
        )
    )
    if title:
        caption_at(
            ax,
            cx - w / 2 + 2.0,
            cy + h / 2 - 2.2,
            title,
            fs=fs,
            color=INK,
            ha="left",
            va="top",
            weight="bold",
        )

In [ ]:
import soundfile as sf  # noqa: E402
import torch  # noqa: E402
import torchaudio  # noqa: E402

from utils.audio import hz_to_y, mel_spectrogram, y_to_hz  # noqa: E402

MEL = mel_spectrogram(P)


def load_segment(audio_path, start_s=0.0, duration_s=3.0, params=P):
    """Trozo mono de un `.wav`, remuestreado a `params.target_sr`."""
    with sf.SoundFile(str(audio_path)) as handle:
        handle.seek(int(start_s * handle.samplerate))
        frames = handle.read(int(duration_s * handle.samplerate), dtype="float32", always_2d=True)
        waveform = torch.from_numpy(frames.mean(axis=1))
        if handle.samplerate != params.target_sr:
            waveform = torchaudio.functional.resample(waveform, handle.samplerate, params.target_sr)
    return waveform


def log_mel(waveform, params=P):
    return np.log(MEL(waveform).numpy() + params.eps)


def draw_spectrogram(ax, image, duration_s, *, params=P, cmap="magma", n_ticks=5, vrange=(2, 99.5)):
    """Espectrograma log-mel con el eje de frecuencia en el espacio mel normalizado."""
    lo, hi = np.percentile(image, vrange)
    ax.imshow(
        image,
        origin="lower",
        aspect="auto",
        cmap=cmap,
        extent=(0, duration_s, 0, 1),
        interpolation="nearest",
        vmin=lo,
        vmax=hi,
    )
    ticks = np.linspace(0, 1, n_ticks)
    ax.set_yticks(ticks, [f"{y_to_hz(np.array([t]), params)[0] / 1000:.1f}" for t in ticks])
    ax.set_ylabel("frecuencia (kHz, eje mel)")
    ax.set_xlabel("tiempo (s)")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return ax


def draw_box(
    ax,
    t0,
    t1,
    low_hz,
    high_hz,
    *,
    params=P,
    color=GOOD,
    label=None,
    lw=1.5,
    dashed=False,
    fs=7.5,
    label_below=False,
):
    """Caja tiempo–frecuencia sobre un espectrograma dibujado por `draw_spectrogram`."""
    y0 = float(hz_to_y(np.array([max(low_hz, params.f_min)]), params)[0])
    y1 = float(hz_to_y(np.array([min(high_hz, params.f_max)]), params)[0])
    ax.add_patch(
        FancyBboxPatch(
            (t0, y0),
            t1 - t0,
            y1 - y0,
            boxstyle="round,pad=0,rounding_size=0.012",
            linewidth=lw,
            edgecolor=color,
            facecolor="none",
            linestyle=(0, (4, 2)) if dashed else "-",
            zorder=4,
        )
    )
    if label:
        ax.text(
            t0,
            (y0 - 0.022) if label_below else (y1 + 0.014),
            label,
            color=color,
            fontsize=fs,
            fontweight="bold",
            va="top" if label_below else "bottom",
            ha="left",
            zorder=5,
        )
    return y0, y1

In [ ]:
from data.annotations import load_annotations  # noqa: E402
from data.species import (  # noqa: E402
    CALL_TYPES,
    LABEL_SEPARATOR,
    VALID_PAIRS,  # noqa: E402
    Species,
)
from prepare_data import EXCLUDED_LABELS, MIN_PAIR_COUNT, select_experiment  # noqa: E402

annotations = load_annotations()
annotations["pair"] = annotations.species + LABEL_SEPARATOR + annotations.call_type

SPECIES_NAME = {s.name.lower(): s.value.replace("_", " ") for s in Species}
SPECIES_LATIN = {
    "aa": "Aotus azarae",
    "ac": "Ateles chamek",
    "as": "Alouatta sara",
    "cc": "Cebus cuscinus",
    "lw": "Leontocebus weddelli",
    "pt": "Plecturocebus toppini",
    "sb": "Saimiri boliviensis",
    "sm": "Sapajus macrocephalus",
}
CALL_NAME = {
    f"{s.name.lower()}/{code}": label.replace("_", " ")
    for s, codes in CALL_TYPES.items()
    for code, label in codes.items()
}
CALL_NAME["lw/trino"] = "trino (tr, tj, tt y tf fusionados)"


# El mismo filtro que materializa el caché (`prepare_data.select_experiment`).
experiment, LABELS = select_experiment(annotations)
experiment = experiment.copy()
experiment["pair"] = experiment["label"]
CLASSES = LABELS.names


def recording_table():
    """Duración de cada .wav de raw/ y si tiene anotación en cleaned/ (leerlas tarda minutos)."""
    cache = DATA_DIR / "recording_durations.csv"
    if cache.exists():
        return pd.read_csv(cache)
    rows = []
    for wav in sorted(RAW_DIR.rglob("*.wav")):
        try:
            info = sf.info(str(wav))
        except Exception:
            continue
        folder = next((p.name for p in wav.parents if "__" in p.name), wav.parent.name)
        cleaned = (CLEANED_DIR / wav.relative_to(RAW_DIR)).with_suffix(".txt")
        rows.append(
            {
                "audio_path": str(wav),
                "species": folder.split("__")[-1].lower(),
                "duration_s": info.duration,
                "has_annotation": cleaned.exists(),
            }
        )
    table = pd.DataFrame(rows)
    table.to_csv(cache, index=False)
    return table


recordings = recording_table()

In [ ]:
# --- Elegir ejemplos reales de forma determinista -----------------------------


def pick_annotation(pair, *, df=None, rank=0, source=None):
    """Anotación de `pair` cuya duración es la `rank`-ésima más cercana a la mediana."""
    table = annotations if df is None else df
    rows = table[table.pair == pair]
    if source is not None:
        rows = rows[rows.audio_path.str.contains(source)]
    if rows.empty:
        raise LookupError(f"sin anotaciones para {pair}")
    order = (rows.duration_s - rows.duration_s.median()).abs().sort_values().index
    return rows.loc[order[min(rank, len(order) - 1)]]


def busiest_segment(pair, window_s=6.0, *, df=None, n_events=6):
    """Ventana de `window_s` con un número de eventos de `pair` cercano a `n_events`."""
    table = annotations if df is None else df
    rows = table[table.pair == pair]
    best = []
    for audio_path, group in rows.groupby("audio_path"):
        for start in group.begin_time_s.to_numpy():
            inside = group[(group.begin_time_s >= start) & (group.end_time_s <= start + window_s)]
            if len(inside):
                best.append(
                    (
                        abs(len(inside) - n_events),
                        -float(inside.duration_s.sum()),
                        audio_path,
                        start,
                    )
                )
    best.sort()
    _, _, audio_path, start = best[0]
    inside = table[
        (table.audio_path == audio_path)
        & (table.end_time_s > start)
        & (table.begin_time_s < start + window_s)
    ]
    return audio_path, float(start), inside


def segment_around(row, pad_ratio=0.9, min_pad_s=0.25, params=P):
    """Inicio y duración de un trozo centrado en la anotación `row`."""
    pad = max(row.duration_s * pad_ratio, min_pad_s)
    start = max(row.begin_time_s - pad, 0.0)
    return start, float(row.duration_s + 2 * pad)

## 5. Detección y evaluación del modelo (OE2)


In [ ]:
# La geometría del DETR sale de las constantes del código, que son las que `train.py` escribe
# en el checkpoint (`PRESETS["detr"]`).
from models.backbone import N_LEVELS, TIME_STRIDE  # noqa: E402
from models.deformable_detr import DIM, N_DECODER_LAYERS, N_QUERIES  # noqa: E402
from train import PRESETS  # noqa: E402

DETR = PRESETS["detr"]
ARCH = {
    "dim": DETR.hparams.get("dim", DIM),
    "n_queries": DETR.hparams.get("n_queries", N_QUERIES),
    "n_levels": N_LEVELS,
    "n_frames": DETR.hparams.get("n_frames", P.n_frames),
    "time_stride": DETR.hparams.get("time_stride", TIME_STRIDE),
    "frontend": DETR.hparams.get("frontend", "pcen"),
    "layers": N_DECODER_LAYERS,
    "n_classes": len(CLASSES),
    "labels": CLASSES,
}

PATCH, FREQ_STRIDE = 16, 10
FREQ_OUT = (P.n_mels - PATCH) // FREQ_STRIDE + 1
TIME_OUT = (ARCH["n_frames"] - PATCH) // ARCH["time_stride"] + 1
# Pirámide {4x, 2x, 1x, 1/2x} sobre el mapa de tokens, como `MultiScalePyramid`.
LEVELS = [(FREQ_OUT * 4, TIME_OUT * 4), (FREQ_OUT * 2, TIME_OUT * 2), (FREQ_OUT, TIME_OUT), (FREQ_OUT // 2, TIME_OUT // 2)]

In [ ]:
# Dos pisos, como en la literatura de detección: arriba el recorrido del dato de punta a
# punta, abajo los dos bloques que no se entienden como una caja --el parcheo del AST y
# la capa del decodificador-- abiertos por dentro. Todas las formas salen de `P` y de
# `ARCH`, así que la figura sigue al checkpoint que se haya entrenado.

fig, ax = canvas(13.4, 8.6)
TOP = 100 * 8.6 / 13.4

FLOW_Y = TOP - 12.5  # eje del piso de arriba
NAME_Y = FLOW_Y + 8.6  # nombre de la etapa
CHIP_Y = NAME_Y + 4.6  # todos entrenables, a la misma altura
SHAPE_Y = FLOW_Y - 7.8  # forma del tensor, va="top"
PANEL_CY, PANEL_H = 15.8, 28.0

mel_shape = f"(1, {P.n_mels}, {P.n_frames})"
token_shape = f"({FREQ_OUT}x{TIME_OUT}, 768)"

# --- Piso 1: el recorrido del dato --------------------------------------------
slab(ax, 7.5, FLOW_Y, 11.0, 10.0)
caption_at(ax, 7.5, NAME_Y, "Espectrograma\nmel", fs=8.0, color=INK, weight="bold")
caption_at(ax, 7.5, SHAPE_Y, mel_shape, fs=6.6, color=INK_2, va="top")

node(ax, 23.0, FLOW_Y, 12.0, 9.0, {"none": "sin front-end", "logmel": "log-mel", "pcen": "PCEN\nentrenable"}[ARCH["frontend"]], fill=SURFACE, edge=BLUE, lw=1.3, fs=8.2)
chip(ax, 23.0, CHIP_Y, "entrenable", GOOD, fs=6.4)
caption_at(ax, 23.0, SHAPE_Y, mel_shape, fs=6.6, color=INK_2, va="top")

node(ax, 40.0, FLOW_Y, 13.0, 9.0, "AST\nafinado entero", fill=SURFACE, edge=BLUE, lw=1.3, fs=8.2)
chip(ax, 40.0, CHIP_Y, "entrenable", GOOD, fs=6.4)
caption_at(ax, 40.0, SHAPE_Y, "ViT-base, AudioSet", fs=6.6, color=INK_2, va="top")

token_strip(ax, 52.0, FLOW_Y, 2.6, 10.0, 7)
caption_at(ax, 52.0, NAME_Y, "Tokens\nlatentes", fs=8.0, color=INK, weight="bold")
caption_at(ax, 52.0, SHAPE_Y, token_shape, fs=6.6, color=INK_2, va="top")

feature_pyramid(ax, 63.0, FLOW_Y, 15.5, 11.0, LEVELS, label_y=SHAPE_Y)
caption_at(ax, 63.0, NAME_Y, "Pirámide\nmultiescala", fs=8.0, color=INK, weight="bold")
chip(ax, 63.0, CHIP_Y, "entrenable", GOOD, fs=6.4)

node(
    ax,
    79.0,
    FLOW_Y,
    14.0,
    9.0,
    "Decodificador\ndeformable",
    fill=SURFACE,
    edge=BLUE,
    lw=1.3,
    fs=8.0,
)
chip(ax, 79.0, CHIP_Y, "entrenable", GOOD, fs=6.4)

detection_slab(
    ax,
    94.0,
    FLOW_Y,
    10.0,
    10.0,
    [(0.32, 0.68, 0.34, 0.20), (0.72, 0.72, 0.24, 0.16), (0.46, 0.30, 0.46, 0.24)],
)
caption_at(ax, 94.0, NAME_Y, "Cajas\netiquetadas", fs=8.0, color=INK, weight="bold")
caption_at(ax, 94.0, SHAPE_Y, "tiempo x frecuencia\n+ clase", fs=6.6, color=INK_2, va="top")

for x0, x1 in ((13.0, 16.8), (29.2, 33.3), (46.6, 50.5), (53.4, 56.8), (69.2, 71.8), (86.2, 88.8)):
    arrow(ax, (x0, FLOW_Y), (x1, FLOW_Y), color=INK, lw=1.3)

# --- Puentes al segundo piso ---------------------------------------------------
zoom_arrow(ax, 40.0, FLOW_Y - 13.2, 6.6, 6.6)
zoom_arrow(ax, 79.0, FLOW_Y - 13.2, 6.6, 6.6)

# --- Piso 2a: cómo el AST parte el espectrograma -------------------------------
detail_panel(ax, 25.5, PANEL_CY, 47.0, PANEL_H, "AST: parcheo y codificación")
inner = PANEL_CY - 2.4

slab(ax, 8.0, inner, 7.0, 9.0)
caption_at(ax, 8.0, inner - 6.6, "mel", fs=6.4, color=INK_2, va="top")

patch_grid(ax, 19.0, inner, 9.0, 9.0, 4, 3)
caption_at(
    ax,
    19.0,
    inner - 6.6,
    f"parches {PATCH}x{PATCH}\npaso ({FREQ_STRIDE}, {ARCH['time_stride']})",
    fs=6.4,
    color=INK_2,
    va="top",
)

token_strip(ax, 28.5, inner, 2.4, 10.0, 6)
node(ax, 35.5, inner, 5.0, 11.0, "Encoder\nViT", fill=PANEL, edge=MUTED, lw=1.1, fs=6.8, wrap=8)
token_strip(ax, 43.0, inner, 2.4, 10.0, 6)
caption_at(ax, 43.0, inner - 6.6, token_shape, fs=6.4, color=INK_2, va="top")

for x0, x1 in ((11.8, 14.2), (23.8, 27.1), (29.9, 32.8), (38.2, 41.6)):
    arrow(ax, (x0, inner), (x1, inner), color=INK, lw=1.1)

# --- Piso 2b: qué hace el decodificador con las queries ------------------------
detail_panel(ax, 74.5, PANEL_CY, 47.0, PANEL_H, "Capa del decodificador deformable")

QUERY_Y = PANEL_CY + 5.6
LAYER_Y = PANEL_CY - 1.2
FEATURE_Y = PANEL_CY - 8.4
JOIN_X = 66.5

token_strip(ax, 59.0, QUERY_Y, 11.0, 2.4, 6, vertical=False)
caption_at(ax, 59.0, QUERY_Y + 3.2, f"{ARCH['n_queries']} object queries", fs=6.4, color=INK_2)

feature_pyramid(ax, 59.0, FEATURE_Y, 13.0, 5.2, LEVELS, label_y=FEATURE_Y - 3.4)
caption_at(ax, 59.0, FEATURE_Y + 4.2, "features multiescala", fs=6.4, color=INK_2)

# Las dos entradas se juntan en una sola flecha: dos puntas encimadas sobre el mismo
# borde se leen como un error de dibujo, no como dos caminos.
for y0 in (QUERY_Y, FEATURE_Y):
    ax.plot([64.8, JOIN_X, JOIN_X], [y0, y0, LAYER_Y], color=INK, lw=1.1, zorder=1)
arrow(ax, (JOIN_X, LAYER_Y), (68.4, LAYER_Y), color=INK, lw=1.1)

node(
    ax,
    76.0,
    LAYER_Y,
    15.0,
    11.5,
    f"Capa x{ARCH['layers']}",
    "auto-atención\natención deformable\nFFN",
    fill=SURFACE,
    edge=BLUE,
    lw=1.2,
    fs=7.4,
    body_fs=6.4,
    wrap=20,
)
node(ax, 87.5, LAYER_Y, 4.2, 10.0, "Cabeza", fill=PANEL, edge=MUTED, lw=1.1, fs=6.6, wrap=7)
token_strip(ax, 93.5, LAYER_Y, 2.3, 8.5, 5)
caption_at(
    ax,
    93.5,
    LAYER_Y - 5.8,
    f"clase ({ARCH['n_classes']}, sigmoide)\ny caja",
    fs=6.4,
    color=INK_2,
    va="top",
)

arrow(ax, (83.6, LAYER_Y), (85.3, LAYER_Y), color=INK, lw=1.1)
arrow(ax, (89.7, LAYER_Y), (92.2, LAYER_Y), color=INK, lw=1.1)

# El refinamiento iterativo: la caja que emite una capa es la referencia de la siguiente.
ax.annotate(
    "",
    xy=(76.0, LAYER_Y + 6.0),
    xytext=(87.5, LAYER_Y + 5.2),
    zorder=2,
    arrowprops=dict(
        arrowstyle="-|>",
        color=MUTED,
        lw=1.0,
        shrinkA=2,
        shrinkB=2,
        connectionstyle="arc3,rad=-0.35",
        mutation_scale=10,
    ),
)
caption_at(ax, 81.5, LAYER_Y + 9.0, "refina la caja de referencia", fs=6.2, color=MUTED)

save(fig, "architecture")
plt.show()

In [ ]:
cell_ms = [P.clip_len_s * 1000 / t for _, t in LEVELS]
durations = experiment.groupby("pair").duration_s.median().sort_values() * 1000

fig, axes = plt.subplots(
    1, 2, figsize=(12.6, 5.6), gridspec_kw={"width_ratios": [0.85, 1.25], "wspace": 0.30}
)

ax = axes[0]
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis("off")
for i, ((f, t), ms) in enumerate(zip(LEVELS, cell_ms)):
    y = 82 - i * 22
    ax.add_patch(
        FancyBboxPatch(
            (4, y - 9),
            56,
            17,
            boxstyle="round,pad=0,rounding_size=1.2",
            facecolor=SURFACE,
            edgecolor=BLUE,
            linewidth=1.3,
        )
    )
    cols = max(4, int(round(40 * t / LEVELS[0][1])))
    rows = max(2, int(round(12 * f / LEVELS[0][0])))
    for c in range(1, cols):
        ax.plot([4 + 56 * c / cols] * 2, [y - 9, y + 8], color=BLUE, linewidth=0.4, alpha=0.45)
    for r in range(1, rows):
        ax.plot([4, 60], [y - 9 + 17 * r / rows] * 2, color=BLUE, linewidth=0.4, alpha=0.45)
    ax.text(
        64,
        y + 2.5,
        f"nivel {i}: {f} x {t}",
        fontsize=9.5,
        fontweight="bold",
        color=INK,
        va="center",
    )
    ax.text(64, y - 3.5, f"una celda = {ms:.0f} ms", fontsize=8.2, color=INK_2, va="center")

ax = axes[1]
ys = np.arange(len(durations))
ax.set_xscale("log")
ax.set_xlim(5, max(durations.max() * 1.9, 4000))
ax.set_ylim(len(durations) - 0.4, -2.6)
for y, (pair, value) in zip(ys, durations.items()):
    ax.plot([5, value], [y, y], color=GRID, linewidth=1.0, zorder=1)
    ax.scatter([value], [y], s=26, color=BLUE, zorder=3)
    ax.text(value * 1.10, y, f"{value:.0f} ms", va="center", fontsize=7, color=INK_2)
for i, ms in enumerate(cell_ms):
    ax.axvline(ms, color=SERIES[1], linewidth=1.2, linestyle=(0, (4, 3)), zorder=2)
    ax.text(
        ms,
        -0.5 - i * 0.62,
        f"nivel {i}: {ms:.0f} ms",
        color=SERIES[1],
        fontsize=7.5,
        ha="left",
        va="center",
    )
ax.set_yticks(ys, durations.index, fontsize=8)
ax.set_xlabel("duración mediana de la clase (ms, escala log)")
style_value_axis(ax, "x")

save(
    fig,
    "multiscale_pyramid",
    data={
        "niveles": [{"f": f, "t": t, "celda_ms": ms} for (f, t), ms in zip(LEVELS, cell_ms)],
        "clase": durations.index,
        "duracion_mediana_ms": durations,
    },
)
plt.show()

In [ ]:
from models.deformable_detr import DeformableAttention  # noqa: E402

attention = DeformableAttention(dim=ARCH["dim"], n_heads=8, n_points=4, n_levels=ARCH["n_levels"])
offsets = attention.offsets.bias.detach().view(8, ARCH["n_levels"], 4, 2).numpy()

REFERENCES = [
    ((0.30, 0.62, 0.06, 0.10), "llamada corta (50 ms)"),
    ((0.62, 0.40, 0.55, 0.34), "llamada larga (1,6 s)"),
]

fig, axes = plt.subplots(1, 2, figsize=(11.4, 5.0), gridspec_kw={"wspace": 0.16})
for ax, (ref, name) in zip(axes, REFERENCES):
    cx, cy, w, h = ref
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.add_patch(Rectangle((0, 0), 1, 1, facecolor=PANEL, edgecolor=BASELINE, linewidth=1.0))
    ax.add_patch(
        Rectangle(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            facecolor=BLUE,
            alpha=0.13,
            edgecolor=BLUE,
            linewidth=1.6,
        )
    )
    ax.scatter([cx], [cy], s=30, color=BLUE, zorder=5)
    points = offsets[:, 0] / 4.0 * np.array([w, h]) * 0.5 + np.array([cx, cy])
    ax.scatter(
        points[..., 0].ravel(),
        points[..., 1].ravel(),
        s=20,
        marker="o",
        facecolor="none",
        edgecolors=SERIES[1],
        linewidths=1.1,
        zorder=4,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.set_title(name, fontsize=10, pad=8)

save(fig, "deformable_sampling")
plt.show()

In [ ]:
fig, ax = canvas(12.4, 5.6)
TOP = 100 * 5.6 / 12.4
cy = TOP * 0.60
blocks = flow_row(
    ax,
    [
        {
            "title": "Query",
            "body": f"{ARCH['n_queries']} vectores\naprendidos",
            "foot": f"({ARCH['n_queries']}, {ARCH['dim']})",
        },
        {
            "title": "Auto-atención",
            "body": "las queries se reparten\nel trabajo entre sí",
            "foot": "+ query_pos en cada capa",
        },
        {
            "title": "Atención deformable",
            "body": "muestrea los 3 niveles\nalrededor de la caja",
            "foot": "8 cabezas x 4 puntos",
        },
        {
            "title": "FFN",
            "body": f"{ARCH['dim']} -> 1024 -> {ARCH['dim']}\ncon LayerNorm residual",
            "foot": "una por capa",
        },
        {
            "title": "Cabezas",
            "body": "clase y delta de caja\nen espacio logit",
            "foot": f"({ARCH['n_classes']}+1) y 4",
        },
    ],
    cy,
    margin=2.5,
    gap=2.6,
    wrap=20,
    fill=SURFACE,
    edge=BLUE,
    lw=1.3,
    fs=8.6,
    body_fs=7.2,
    foot_fs=6.8,
)
h = blocks[0][3]
caption_at(ax, blocks[0][0], cy + h / 2 + 3.2, "x 6 capas", fs=8, color=INK_2)
elbow(
    ax,
    [
        (blocks[-1][0], cy - h / 2),
        (blocks[-1][0], cy - h / 2 - 6),
        (blocks[2][0], cy - h / 2 - 6),
        (blocks[2][0], cy - h / 2 - 1),
    ],
    color=SERIES[1],
)
caption_at(
    ax,
    blocks[2][0],
    cy - h / 2 - 8.5,
    "refinamiento iterativo",
    fs=7.6,
    color=SERIES[1],
    wrap=90,
    va="top",
)
save(fig, "decoder_layer")
plt.show()

In [ ]:
from scipy.optimize import linear_sum_assignment  # noqa: E402

from models.criterion import HungarianMatcher  # noqa: E402

truth_boxes = torch.tensor(
    [[0.20, 0.62, 0.10, 0.14], [0.52, 0.58, 0.14, 0.18], [0.80, 0.30, 0.20, 0.22]],
    dtype=torch.float32,
)
truth_labels = torch.tensor([0, 0, 1])
predicted_boxes = torch.tensor(
    [
        [0.21, 0.63, 0.12, 0.15],  # buena sobre la primera
        [0.50, 0.57, 0.20, 0.26],  # laxa sobre la segunda
        [0.79, 0.31, 0.19, 0.20],  # buena sobre la tercera
        [0.35, 0.20, 0.10, 0.10],  # fondo
        [0.65, 0.80, 0.16, 0.12],  # fondo
    ],
    dtype=torch.float32,
)
logits = torch.full((5, 3), -2.0)
logits[0, 0] = logits[2, 1] = 3.0
logits[1, 0] = 1.2
logits[3, 2] = logits[4, 2] = 2.5  # el canal de no-objeto

matcher = HungarianMatcher()
outputs = {"pred_logits": logits[None], "pred_boxes": predicted_boxes[None]}
targets = [{"labels": truth_labels, "boxes": truth_boxes}]
query_index, target_index = matcher(outputs, targets)[0]

probabilities = logits.softmax(-1)
cost_class = -probabilities[:, truth_labels]
cost_bbox = torch.cdist(predicted_boxes, truth_boxes, p=1)
from torchvision.ops import box_convert, generalized_box_iou  # noqa: E402

cost_iou = -generalized_box_iou(
    box_convert(predicted_boxes, "cxcywh", "xyxy"), box_convert(truth_boxes, "cxcywh", "xyxy")
)
cost = (
    matcher.cost_class * cost_class + matcher.cost_bbox * cost_bbox + matcher.cost_iou * cost_iou
).numpy()

fig, axes = plt.subplots(
    1, 2, figsize=(12.0, 5.0), gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.24}
)

ax = axes[0]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.add_patch(Rectangle((0, 0), 1, 1, facecolor=PANEL, edgecolor=BASELINE, linewidth=1.0))
for i, box in enumerate(truth_boxes.numpy()):
    cx, cy, w, h = box
    ax.add_patch(
        Rectangle((cx - w / 2, cy - h / 2), w, h, facecolor="none", edgecolor=GOOD, linewidth=2.0)
    )
    ax.text(cx - w / 2, cy + h / 2 + 0.02, f"g{i}", color=GOOD, fontsize=8.5, fontweight="bold")
for j, box in enumerate(predicted_boxes.numpy()):
    cx, cy, w, h = box
    matched = j in query_index.tolist()
    ax.add_patch(
        Rectangle(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            facecolor="none",
            edgecolor=BLUE if matched else MUTED,
            linewidth=1.6 if matched else 1.1,
            linestyle="-" if matched else (0, (3, 2)),
        )
    )
    ax.text(
        cx + w / 2,
        cy - h / 2 - 0.03,
        f"q{j}",
        color=BLUE if matched else MUTED,
        fontsize=8.5,
        ha="right",
    )
ax.set_xticks([])
ax.set_yticks([])
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(False)

ax = axes[1]
image = ax.imshow(cost, cmap="Blues_r", aspect="auto")
for j in range(cost.shape[0]):
    for i in range(cost.shape[1]):
        chosen = (j, i) in list(zip(query_index.tolist(), target_index.tolist()))
        ax.text(
            i,
            j,
            f"{cost[j, i]:.2f}",
            ha="center",
            va="center",
            fontsize=8.5,
            color=INK if cost[j, i] > cost.min() + (cost.max() - cost.min()) * 0.45 else SURFACE,
            fontweight="bold" if chosen else "normal",
        )
        if chosen:
            ax.add_patch(
                Rectangle(
                    (i - 0.5, j - 0.5), 1, 1, facecolor="none", edgecolor=SERIES[1], linewidth=2.4
                )
            )
ax.set_xticks(
    range(3),
    [f"g{i}\n{CLASSES[label]}" for i, label in enumerate(truth_labels.tolist())],
    fontsize=8,
)
ax.set_yticks(range(5), [f"q{j}" for j in range(5)], fontsize=8)
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(False)

save(fig, "hungarian_matching")
plt.show()

In [ ]:
from core.config import MAX_DETECTIONS  # noqa: E402
from evaluation.metrics import MATCH_IOU  # noqa: E402
from train import PRESETS  # noqa: E402

CONFIG = PRESETS["detr"].config

fig, ax = canvas(13.2, 4.4)
TOP = 100 * 4.4 / 13.2
cy = TOP * 0.60
boxes = flow_row(
    ax,
    [
        {
            "title": "Caché de tensores",
            "body": "train.pt / val.pt\nmel de potencia + cajas",
            "foot": "`SpectrogramDataset`",
        },
        {
            "title": "Aumento",
            "body": f"cajas: jitter ±{CONFIG.jitter.time_scale:.0%}\naudio: ganancia, fondo, corrimiento, copy-paste",
            "foot": "sólo en entrenamiento",
        },
        {
            "title": "Modelo",
            "body": f"lotes de {CONFIG.batch_size}\nAdamW lr {CONFIG.learning_rate:g}, wd {CONFIG.weight_decay:g}",
            "foot": f"OneCycle, {CONFIG.epochs} épocas",
        },
        {
            "title": "SetCriterion",
            "body": "húngaro + focal, L1 y GIoU\nen las 6 capas",
            "foot": "sin canal de no-objeto",
        },
        {
            "title": "Validación por época",
            "body": f"recall y precisión a IoU {MATCH_IOU:g}\ncon score ≥ {CONFIG.score_threshold:g}",
            "foot": f"sin NMS, tope {MAX_DETECTIONS} por ventana",
        },
        {
            "title": "Selección",
            "body": f"mejor mAP@{MATCH_IOU:g} en val\nsobre toda la curva",
            "foot": "guarda best.pt",
        },
    ],
    cy,
    margin=2.0,
    gap=2.0,
    wrap=20,
    fill=SURFACE,
    edge=BLUE,
    lw=1.3,
    fs=8.4,
    body_fs=7.0,
    foot_fs=6.7,
)
h = boxes[0][3]
elbow(
    ax,
    [
        (boxes[-1][0], cy - h / 2),
        (boxes[-1][0], cy - h / 2 - 5),
        (boxes[2][0], cy - h / 2 - 5),
        (boxes[2][0], cy - h / 2 - 1),
    ],
    color=SERIES[1],
)
caption_at(ax, boxes[2][0], cy - h / 2 - 7.5, "una época", fs=7.4, color=SERIES[1], va="top")
save(fig, "training_pipeline")
plt.show()

In [ ]:
from core.config import NMS_IOU  # noqa: E402
from inference.predictor import BATCH_SIZE  # noqa: E402

fig, ax = canvas(13.2, 4.4)
TOP = 100 * 4.4 / 13.2
cy = TOP * 0.60
boxes = flow_row(
    ax,
    [
        {"title": "Grabación .wav", "body": "de cualquier duración", "foot": "`inference.predictor.predict`"},
        {
            "title": "Ventaneo",
            "body": f"clips de {P.clip_len_s:g} s cada {P.clip_hop_s:g} s",
            "foot": "`window_starts`",
        },
        {
            "title": "Mel de potencia",
            "body": "el mismo transform\nque en entrenamiento",
            "foot": f"(1, {P.n_mels}, {P.n_frames})",
        },
        {
            "title": "Modelo",
            "body": f"{ARCH['n_queries']} candidatos\npor ventana",
            "foot": f"en lotes de {BATCH_SIZE}",
        },
        {
            "title": "Posproceso",
            "body": f"por ventana, el NMS del modelo\nentre ventanas, IoU {NMS_IOU:g} + IoMin 0,8",
            "foot": "`postprocess` + `suppress_nested`",
        },
        {
            "title": "Tabla de Raven",
            "body": "tiempo en s y\nfrecuencia en Hz",
            "foot": "`raven_table`",
        },
    ],
    cy,
    margin=2.0,
    gap=2.0,
    wrap=20,
    fill=SURFACE,
    edge=SERIES[2],
    arrow_color=SERIES[2],
    lw=1.3,
    fs=8.4,
    body_fs=7.0,
    foot_fs=6.7,
)
save(fig, "inference_pipeline")
plt.show()

In [ ]:
# Curvas de entrenamiento de cada corrida en runs/: `metrics.jsonl` del Trainer (FRCNN, DETR)
# o `results.csv` de Ultralytics (YOLO). Sin corridas, se salta.
runs = {}
for run_dir in sorted(RUNS_DIR.iterdir()):
    if (run_dir / "metrics.jsonl").is_file():
        rows = [json.loads(line) for line in (run_dir / "metrics.jsonl").read_text().splitlines() if line.strip()]
        runs[run_dir.name] = pd.DataFrame(
            {
                "epoch": [r["epoch"] for r in rows],
                "train": [r["train"]["total"] for r in rows],
                "val": [r["val_loss"]["total"] for r in rows],
                "metric": [r["val"]["map_30"] for r in rows],
                "metric_name": "mAP@0.3 (val)",
            }
        )
    elif (run_dir / "results.csv").is_file():
        table = pd.read_csv(run_dir / "results.csv")
        table.columns = [c.strip() for c in table.columns]
        runs[run_dir.name] = pd.DataFrame(
            {
                "epoch": table["epoch"],
                "train": table["train/box_loss"] + table["train/cls_loss"],
                "val": table["val/box_loss"] + table["val/cls_loss"],
                "metric": table["metrics/mAP50(B)"],
                "metric_name": "mAP@0.5 (val, Ultralytics)",
            }
        )

if not runs:
    print("sin corridas con metrics.jsonl ni results.csv en runs/")
else:
    fig, axes = plt.subplots(len(runs), 2, figsize=(11.0, 3.2 * len(runs)), squeeze=False)
    for (name, table), (ax_loss, ax_metric) in zip(runs.items(), axes):
        ax_loss.plot(table.epoch, table.train, color=BLUE, label="train")
        ax_loss.plot(table.epoch, table.val, color=SERIES[1], label="val")
        ax_loss.set_title(f"{name}: pérdida", pad=6)
        ax_loss.legend(loc="upper right")
        ax_metric.plot(table.epoch, table.metric, color=SERIES[2])
        best = int(table.metric.idxmax())
        ax_metric.scatter([table.epoch[best]], [table.metric[best]], s=30, color=SERIES[2], zorder=4)
        ax_metric.set_title(f"{name}: {table.metric_name.iloc[0]}", pad=6)
        for ax in (ax_loss, ax_metric):
            ax.set_xlabel("época")
            style_value_axis(ax, "y")
    fig.subplots_adjust(hspace=0.6, wspace=0.25)
    save(fig, "training_curves", data={name: t.drop(columns="metric_name") for name, t in runs.items()})
    plt.show()

In [ ]:
# La comparación vigente: el .json más reciente de runs/comparacion/ (`compare_models.py`).
comparisons = sorted(COMPARISON_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime)
COMPARISON = json.loads(comparisons[-1].read_text()) if comparisons else None
if COMPARISON is None:
    print("sin comparación en runs/comparacion/: corré ./evaluation.sh")
else:
    print(comparisons[-1].name, "|", ", ".join(m["model"] for m in COMPARISON["models"]))
    MODELS = COMPARISON["models"]
    MODEL_COLOR = {m["model"]: SERIES[i % len(SERIES)] for i, m in enumerate(MODELS)}
    protocol = COMPARISON["protocol"]

    # mAP libre de umbral
    fig, ax = plt.subplots(figsize=(7.0, 0.6 * len(MODELS) + 1.2))
    ys = np.arange(len(MODELS))
    for y, m in zip(ys, MODELS):
        barh(ax, [y], [m["map_30"] or 0.0], MODEL_COLOR[m["model"]], height=0.58)
        ax.text((m["map_30"] or 0.0) + 0.01, y, f"{m['map_30']:.3f}", va="center", fontsize=8, color=INK_2)
    ax.set_xlim(0, 1)
    ax.set_ylim(len(MODELS) - 0.4, -0.7)
    ax.set_yticks(ys, [m["model"] for m in MODELS])
    ax.set_xlabel(f"mAP@{protocol['iou']:g} en test ({len(COMPARISON['detection_classes'])} clases de detección)")
    style_value_axis(ax, "x")
    save(fig, "comparison_map", data={"modelo": [m["model"] for m in MODELS], "map": [m["map_30"] for m in MODELS], "protocolo": protocol})
    plt.show()

In [ ]:
# Punto pareado: umbral elegido en val con una precisión mínima, medido en test con IC 95 %
# por bootstrap de grabaciones. Un panel por precisión mínima. La comparación guardada trae sólo
# el primer umbral sin intervalo: acá se recalculan desde los volcados de cada modelo.
from evaluation.evaluator import RawPredictions  # noqa: E402
from evaluation.protocol import MIN_PRECISIONS, equalize, match, pair  # noqa: E402


def dump_of(name: str, split: str) -> RawPredictions | None:
    found = [p for p in RUNS_DIR.glob(f"*/{name}_{split}_predictions.pt")]
    return RawPredictions.load(found[0]) if found else None


if COMPARISON is not None:
    PAIRED = {}
    for m in MODELS:
        val, test = dump_of(m["model"], "val"), dump_of(m["model"], "test")
        if val is None or test is None:
            print(f"{m['model']}: sin volcados de val y test en runs/")
            continue
        matched_val = match(equalize(val.predictions, protocol["max_det"]), val.truth, val.n_images)
        matched = match(equalize(test.predictions, protocol["max_det"]), test.truth, test.n_images)
        PAIRED[m["model"]] = [pair(matched_val, matched, test.recordings, p)._asdict() for p in MIN_PRECISIONS]
    points = list(MIN_PRECISIONS)
    fig, axes = plt.subplots(1, len(points), figsize=(5.6 * len(points), 0.7 * len(MODELS) + 1.8), squeeze=False)
    rows = []
    for ax, position in zip(axes[0], range(len(points))):
        min_precision = points[position]
        ys = np.arange(len(MODELS))
        for y, m in zip(ys, MODELS):
            if m["model"] not in PAIRED:
                continue
            p = PAIRED[m["model"]][position]
            color = MODEL_COLOR[m["model"]]
            if p["test"] is None:
                ax.text(0.02, y, "ningún umbral llega", va="center", fontsize=8, color=MUTED)
                continue
            for metric, offset, marker in (("recall", -0.16, "o"), ("precision", 0.16, "s")):
                lo, hi = p["interval"][metric]
                ax.plot([lo, hi], [y + offset] * 2, color=color, linewidth=2.2, solid_capstyle="round", alpha=0.45)
                ax.scatter([getattr(p["test"], metric)], [y + offset], s=34, color=color, marker=marker, zorder=4)
            ax.text(1.01, y, f"umbral {p['threshold']:.2f}", va="center", fontsize=7.6, color=INK_2)
            rows.append({"modelo": m["model"], "precision_minima": min_precision, **{k: getattr(p["test"], k) for k in ("threshold", "recall", "precision")}, "ic": p["interval"]})
        ax.set_xlim(0, 1)
        ax.set_ylim(len(MODELS) - 0.4, -0.8)
        ax.set_yticks(ys, [m["model"] for m in MODELS])
        ax.set_title(f"precisión ≥ {min_precision:.2f} en val", pad=8)
        ax.set_xlabel("en test: ● recall   ■ precisión   (IC 95 %)")
        style_value_axis(ax, "x")
    fig.subplots_adjust(wspace=0.55)
    save(fig, "comparison_paired", data={"puntos": rows, "protocolo": protocol})
    plt.show()

In [ ]:
# Por clase, al primer punto de operación: recall en test y AP sobre toda la curva.
if COMPARISON is not None and any(m["per_class"] for m in MODELS):
    reference = [m for m in MODELS if m["per_class"]][0]
    classes = [r["name"] for r in sorted(reference["per_class"], key=lambda r: -r["n_gt"])]
    n_gt = {r["name"]: r["n_gt"] for r in reference["per_class"]}
    ys = np.arange(len(classes))
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 0.3 * len(classes) + 1.6), sharey=True, gridspec_kw={"wspace": 0.08})
    for ax, key in zip(axes, ("recall", "ap")):
        for m in MODELS:
            values = {r["name"]: r[key] for r in m["per_class"]}
            ax.scatter([values.get(c) if values.get(c) is not None else np.nan for c in classes], ys, s=26, color=MODEL_COLOR[m["model"]], label=m["model"], zorder=3)
        for y in ys:
            ax.plot([0, 1], [y, y], color=GRID, linewidth=0.8, zorder=1)
        ax.set_xlim(0, 1)
        ax.set_ylim(len(classes) - 0.4, -0.8)
        ax.set_xlabel({"recall": "recall en test al primer punto de operación", "ap": f"AP@{protocol['iou']:g} en test"}[key])
        style_value_axis(ax, "x")
    axes[0].set_yticks(ys, [f"{c}  ({n_gt[c]:,})" for c in classes], fontsize=7.6)
    axes[0].legend(loc="lower left")
    save(fig, "comparison_per_class", data={"clase": classes, "cajas_gt": [n_gt[c] for c in classes], "modelos": [{"modelo": m["model"], "por_clase": m["per_class"]} for m in MODELS]})
    plt.show()

In [ ]:
# Barrido del umbral de score en test para cada modelo de la comparación, desde sus volcados,
# con el mismo protocolo (tope de detecciones, IoU) y el umbral elegido en val marcado.
from evaluation.protocol import THRESHOLD_GRID, operating_point  # noqa: E402

if COMPARISON is not None:
    sweeps = {}
    for m in MODELS:
        name = m["model"]
        dump = dump_of(name, "test")
        if dump is None:
            print(f"{name}: sin volcado de test en runs/")
            continue
        matched = match(equalize(dump.predictions, protocol["max_det"]), dump.truth, dump.n_images)
        curve = [operating_point(matched, t) for t in THRESHOLD_GRID]
        sweeps[name] = pd.DataFrame({"umbral": THRESHOLD_GRID, "recall": [p.recall for p in curve], "precision": [p.precision for p in curve]})

    if sweeps:
        fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4), gridspec_kw={"wspace": 0.25})
        for name, table in sweeps.items():
            color = MODEL_COLOR[name]
            axes[0].plot(table.umbral, table.recall, color=color, label=name)
            axes[0].plot(table.umbral, table.precision, color=color, linestyle=(0, (4, 3)))
            axes[1].plot(table.recall, table.precision, color=color, label=name)
            chosen = [m["threshold"] for m in MODELS if m["model"] == name and m["threshold"] is not None]
            if chosen:
                t = chosen[0]
                row = table.iloc[(table.umbral - t).abs().argmin()]
                axes[0].axvline(t, color=color, linewidth=0.9, alpha=0.5)
                axes[1].scatter([row.recall], [row.precision], s=44, color=color, zorder=4, edgecolor=SURFACE)
        axes[0].set_xlabel("umbral de score")
        axes[0].set_ylabel("recall (línea llena) · precisión (a trazos)")
        axes[0].set_xlim(0, 1)
        axes[0].set_ylim(0, 1)
        axes[0].legend(loc="lower left")
        axes[1].set_xlabel("recall")
        axes[1].set_ylabel("precisión")
        axes[1].set_xlim(0, 1)
        axes[1].set_ylim(0, 1)
        for ax in axes:
            style_value_axis(ax, "y")
        save(fig, "score_sweep", data={name: t for name, t in sweeps.items()})
        plt.show()